In [7]:
import os
import re
import torch
import librosa
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor, pipeline

In [9]:
# --- 1. 配置与初始化 ---
INPUT_WAV = "evaluation/airport.wav"  # 输入的长音频文件
OUTPUT_TXT = "detection_results.txt"
KEYWORD = "CA151"

WINDOW_SIZE_SEC = 1.0  # 窗口长度 1s
STRIDE_SEC = 0.5       # 步长 0.5s
SAMPLE_RATE = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 加载 AST 语音检测模型
ast_model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = AutoFeatureExtractor.from_pretrained(ast_model_name)
ast_model = AutoModelForAudioClassification.from_pretrained(ast_model_name).to(device)
ast_model.eval()

# 加载 Distil-Whisper 转录模型
distil_model_id = "distil-whisper/distil-large-v3"
transcribe_pipe = pipeline(
    "automatic-speech-recognition",
    model=distil_model_id,
    torch_dtype=torch_dtype,
    device=0 if device == "cuda" else -1,
    model_kwargs={"attn_implementation": "sdpa"} if device == "cuda" else {}
)

# --- 2. 核心功能函数 ---

def is_speech_window(audio_buffer, threshold=0.6, min_rms=0.01):
    """
    优化后的检测函数
    :param threshold: 语音类标签的最小概率总和 (0.0 到 1.0)
    :param min_rms: 最小能量门限，低于此值直接判定为静音
    """
    # --- 步骤 A: 能量过滤 (初步屏蔽底噪) ---
    rms = np.sqrt(np.mean(audio_buffer**2))
    if rms < min_rms:
        return False, 0.0

    # --- 步骤 B: AST 深度检测 ---
    inputs = ast_extractor(audio_buffer, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = ast_model(**inputs)
    
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # 定义我们关心的“真语音”标签索引 (在 AudioSet 中)
    # 0: Speech, 1: Child speech, 2: Conversation, 3: Babbling
    speech_indices = [0, 1, 2, 3] 
    
    # 计算这些语音标签的概率总和
    speech_prob = float(probs[0, speech_indices].sum())
    
    # 只有概率超过阈值才判定为有效语音
    return speech_prob >= threshold, speech_prob

def merge_intervals(intervals):
    """合并重叠或相邻的时间片段"""
    if not intervals: return []
    intervals.sort(key=lambda x: x[0])
    merged = [list(intervals[0])]
    for curr_start, curr_end in intervals[1:]:
        prev_end = merged[-1][1]
        if curr_start <= prev_end:  # 有重叠
            merged[-1][1] = max(prev_end, curr_end)
        else:
            merged.append([curr_start, curr_end])
    return merged

# --- 3. 执行流程 ---

print(f"正在加载长音频: {INPUT_WAV} ...")
audio_full, _ = librosa.load(INPUT_WAV, sr=SAMPLE_RATE)
duration_sec = len(audio_full) / SAMPLE_RATE

# A. 滑动窗口检测
print(f"开始滑动窗口检测 (步长: {STRIDE_SEC}s)...")
speech_windows = []
for start_sec in np.arange(0, duration_sec - WINDOW_SIZE_SEC, STRIDE_SEC):
    start_idx = int(start_sec * SAMPLE_RATE)
    end_idx = int(start_sec + WINDOW_SIZE_SEC) * SAMPLE_RATE
    buffer = audio_full[start_idx:end_idx]
    
    # 调高阈值（例如 0.8）可以变得非常“高冷”，只识别清晰的人声
    is_speech, confidence = is_speech_window(buffer, threshold=0.4, min_rms=0.02)
    
    if is_speech:
        speech_windows.append((start_sec, start_sec + WINDOW_SIZE_SEC))

# B. 合并连续片段
merged_segments = merge_intervals(speech_windows)
print(f"检测完成，共提取到 {len(merged_segments)} 条连续语音片段。")

# C. 转录与分类
results_with_keyword = []
results_others = []
target_pattern = re.compile(r"C[.\s]?A[- \s]?151", re.IGNORECASE)

for idx, (start, end) in enumerate(merged_segments):
    start_idx = int(start * SAMPLE_RATE)
    end_idx = int(end * SAMPLE_RATE)
    segment_audio = audio_full[start_idx:end_idx]
    
    # 执行转录
    # 注意：pipeline 可以直接接收 numpy 数组
    transcription = transcribe_pipe(segment_audio)["text"].strip()
    
    has_keyword = bool(target_pattern.search(transcription))
    
    timestamp_str = f"[{start:0.2f}s - {end:0.2f}s]"
    entry = f"{timestamp_str}: {transcription}"
    
    if has_keyword:
        results_with_keyword.append(entry)
        print(f"✅ 捕获成功: {entry}")
    else:
        results_others.append(entry)

# --- 4. 保存结果 ---
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(f"=== 包含关键词 '{KEYWORD}' 的片段 ===\n")
    if not results_with_keyword: f.write("无\n")
    for r in results_with_keyword: f.write(r + "\n")
    
    f.write(f"\n=== 其他语音片段 ===\n")
    if not results_others: f.write("无\n")
    for r in results_others: f.write(r + "\n")

print(f"\n任务完成！结果已保存至: {OUTPUT_TXT}")

Loading weights: 100%|██████████| 539/539 [00:00<00:00, 2002.96it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight] 


正在加载长音频: evaluation/airport.wav ...
开始滑动窗口检测 (步长: 0.5s)...
检测完成，共提取到 218 条连续语音片段。
✅ 捕获成功: [10.00s - 19.00s]: Your attention please. Passengers for flight CA 151, we are now ready for boarding at Gate B-28.
✅ 捕获成功: [22.00s - 30.00s]: Your attention please. Passengers for Flight CA 151, we are now ready for boarding at Gate B-28.
✅ 捕获成功: [32.00s - 40.00s]: Your attention please. Passengers for flight CA-151, we are now ready for boarding at Gate B-28.
✅ 捕获成功: [42.00s - 50.00s]: Your attention please. Passengers for flight CA 151. We are now ready for boarding at Gate B-28.
✅ 捕获成功: [52.00s - 60.50s]: Attention please. Flight CA 151 to Beijing is now boarding priority passengers and those requiring special assistance.
✅ 捕获成功: [61.00s - 70.50s]: Attention please. Flight CA 151 to Beijing is now boarding priority passengers and those requiring special assistance.
✅ 捕获成功: [71.00s - 80.50s]: Attention please. Flight CA 151 to Beijing is now boarding priority passengers and those requiring spec